# **Install dependency**

In [0]:
%pip install --quiet "psycopg[binary]"

## **PostgreSQL initial snapshot extractor**

In [0]:
# ============================================================
# BANGLAPAY | PHASE 1
# POSTGRESQL BRONZE INITIAL SNAPSHOT
#
# Environments : DEV (20%) / PROD (100%)
# Source       : Neon PostgreSQL
# Tables       : 22
# Landing      : Unity Catalog Managed Volume
# Format       : NDJSON
#
# Source access: READ ONLY
# Snapshot     : INITIAL_SNAPSHOT
#
# IMPORTANT:
# - Run DEV mode only in the DEV Databricks workspace.
# - Run PROD mode only in the PROD Databricks workspace.
# - The environment widget does not detect the workspace.
# - Do not start the Bronze pipeline while extraction is running.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import json
import uuid
import base64

from datetime import date, datetime, timezone
from decimal import Decimal
from uuid import UUID

import psycopg

from psycopg import sql
from psycopg.rows import dict_row

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    TimestampType
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

dbutils.widgets.dropdown(
    "environment",
    "dev",
    ["dev", "prod"],
    "Target Environment"
)

dbutils.widgets.text(
    "confirm_prod",
    "",
    "Production Confirmation"
)

ENVIRONMENT = dbutils.widgets.get("environment").strip().lower()

if ENVIRONMENT not in ("dev", "prod"):
    raise ValueError(
        f"Invalid environment: {ENVIRONMENT}"
    )

SAMPLE_PERCENT = {
    "dev": 20,
    "prod": 100
}[ENVIRONMENT]

if ENVIRONMENT == "prod":
    confirmation = dbutils.widgets.get(
        "confirm_prod"
    ).strip()

    if confirmation != "RUN_PROD_INITIAL_SNAPSHOT":
        raise RuntimeError(
            "Production snapshot blocked. "
            "Set confirm_prod to RUN_PROD_INITIAL_SNAPSHOT."
        )

EXPECTED_TABLE_COUNT = 22

DATABASE_NAME = "banglapay"

SECRET_SCOPE = "banglapay-ingestion"

VOLUME_NAME = "banglapay.ops.batch_landing"

VOLUME_ROOT = (
    "/Volumes/banglapay/ops/batch_landing/postgres"
)

MANIFEST_TABLE = (
    "banglapay.ops.postgres_batch_manifest"
)

FETCH_SIZE = 1000

SNAPSHOT_TYPE = "INITIAL_SNAPSHOT"

BATCH_ID = str(uuid.uuid4())

EXTRACTED_AT = datetime.now(timezone.utc)


# Environment / sampling guard
if ENVIRONMENT == "dev":
    assert SAMPLE_PERCENT == 20

elif ENVIRONMENT == "prod":
    assert SAMPLE_PERCENT == 100

else:
    raise ValueError(
        f"Unsupported environment: {ENVIRONMENT}"
    )

print("=" * 70)
print("BANGLAPAY POSTGRESQL INITIAL SNAPSHOT")
print("=" * 70)
print(f"Environment       : {ENVIRONMENT.upper()}")
print(f"Sample percentage : {SAMPLE_PERCENT}%")
print(f"Expected tables   : {EXPECTED_TABLE_COUNT}")
print(f"Snapshot type     : {SNAPSHOT_TYPE}")
print(f"Batch ID          : {BATCH_ID}")
print(f"Landing volume    : {VOLUME_NAME}")
print()
print(
    "WORKSPACE CHECK: Confirm that this notebook is running "
    f"in the {ENVIRONMENT.upper()} Databricks workspace."
)

if ENVIRONMENT == "prod":
    print(
        "PRODUCTION MODE: This run will extract 100% of the "
        "22 PostgreSQL source tables."
    )


# ============================================================
# 2. VALIDATE TARGET AND PREVENT DUPLICATE INITIAL SNAPSHOTS
# ============================================================

# Confirm that the expected Unity Catalog Volume is accessible.
try:
    dbutils.fs.ls(VOLUME_ROOT)
except Exception as exc:
    raise RuntimeError(
        "The PostgreSQL landing Volume is not accessible. "
        f"Verify {VOLUME_NAME} and the current workspace permissions."
    ) from exc


# Create the append-only manifest table if it does not exist.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MANIFEST_TABLE} (

    batch_id STRING,
    environment STRING,

    source_system STRING,
    source_database STRING,

    source_schema STRING,
    source_table STRING,

    bronze_table STRING,

    snapshot_type STRING,
    sample_percent INT,

    extracted_rows BIGINT,

    file_path STRING,

    extracted_at TIMESTAMP

)
USING DELTA

COMMENT 'Append-only PostgreSQL Bronze batch extraction manifest.'
""")


# An initial snapshot is intended to be executed once per environment.
# DEV and PROD have separate Unity Catalog storage in this project.
existing_initial_snapshot = (
    spark.table(MANIFEST_TABLE)
    .filter(
        (F.col("environment") == ENVIRONMENT)
        & (F.col("snapshot_type") == SNAPSHOT_TYPE)
    )
    .limit(1)
    .count()
)

if existing_initial_snapshot > 0:
    raise RuntimeError(
        f"An {SNAPSHOT_TYPE} batch is already registered for "
        f"environment '{ENVIRONMENT}'. "
        "This notebook is a one-time initial snapshot. "
        "Do not rerun it to avoid duplicate Bronze data."
    )

print(
    f"Manifest check     : PASS "
    f"(no prior {ENVIRONMENT.upper()} initial snapshot)"
)


# ============================================================
# 3. LOAD POSTGRESQL SECRETS
# ============================================================

JDBC_URL = dbutils.secrets.get(
    SECRET_SCOPE,
    "NEON_JDBC_URL"
).strip()

PG_USERNAME = dbutils.secrets.get(
    SECRET_SCOPE,
    "NEON_USERNAME"
).strip()

PG_PASSWORD = dbutils.secrets.get(
    SECRET_SCOPE,
    "NEON_PASSWORD"
).strip()


if JDBC_URL.startswith("jdbc:"):
    PG_URL = JDBC_URL[len("jdbc:"):]

elif JDBC_URL.startswith(
    ("postgresql://", "postgres://")
):
    PG_URL = JDBC_URL

else:
    raise ValueError(
        "NEON_JDBC_URL has an unsupported format."
    )


if not all([PG_URL, PG_USERNAME, PG_PASSWORD]):
    raise RuntimeError(
        "PostgreSQL connection configuration is incomplete."
    )


print("PostgreSQL secrets : loaded")


# ============================================================
# 4. JSON SERIALIZATION
#
# Preserve source values without business-level transformations.
# ============================================================

def json_safe(value):

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if isinstance(value, Decimal):
        return str(value)

    if isinstance(value, UUID):
        return str(value)

    if isinstance(value, memoryview):
        value = value.tobytes()

    if isinstance(value, bytes):
        return {
            "__encoding": "base64",
            "value": base64.b64encode(value).decode("ascii")
        }

    raise TypeError(
        "Unsupported PostgreSQL value type: "
        f"{type(value).__name__}"
    )


# ============================================================
# 5. DISCOVER ALL TABLES AND PRIMARY KEYS
# ============================================================

TABLE_INVENTORY_QUERY = """
SELECT
    t.table_schema,
    t.table_name,

    array_agg(
        kcu.column_name::text
        ORDER BY kcu.ordinal_position
    ) FILTER (
        WHERE kcu.column_name IS NOT NULL
    ) AS primary_key_columns

FROM information_schema.tables t

LEFT JOIN information_schema.table_constraints tc

    ON tc.table_schema = t.table_schema
   AND tc.table_name = t.table_name
   AND tc.constraint_type = 'PRIMARY KEY'

LEFT JOIN information_schema.key_column_usage kcu

    ON kcu.constraint_catalog = tc.constraint_catalog
   AND kcu.constraint_schema = tc.constraint_schema
   AND kcu.constraint_name = tc.constraint_name
   AND kcu.table_schema = tc.table_schema
   AND kcu.table_name = tc.table_name

WHERE t.table_type = 'BASE TABLE'

  AND t.table_schema NOT IN (
      'pg_catalog',
      'information_schema'
  )

GROUP BY
    t.table_schema,
    t.table_name

ORDER BY
    t.table_schema,
    t.table_name
"""


# ============================================================
# 6. DETERMINISTIC PRIMARY KEY SAMPLING
#
# Same primary key always selects the same DEV sample.
# PROD uses 100%, so every row passes the predicate.
# ============================================================

def build_sample_predicate(primary_keys):

    if not primary_keys:
        raise ValueError(
            "Cannot build a deterministic sample without a primary key."
        )

    key_parts = [
        sql.SQL("CAST({} AS TEXT)").format(
            sql.Identifier(column)
        )
        for column in primary_keys
    ]

    pk_expression = sql.SQL(
        "concat_ws('|', {})"
    ).format(
        sql.SQL(", ").join(key_parts)
    )

    predicate = sql.SQL("""
        mod(
            abs(
                (
                    'x' || substr(md5({}), 1, 8)
                )::bit(32)::bigint
            ),
            100
        ) < %s
    """).format(pk_expression)

    return predicate


# ============================================================
# 7. EXTRACT TABLES TO PENDING FILES
#
# Pending files end in .ndjson.pending and are excluded by the
# Bronze Auto Loader glob filter '*.ndjson'.
# They are renamed to .ndjson only after all 22 tables finish.
# ============================================================

os.makedirs(
    VOLUME_ROOT,
    exist_ok=True
)

manifest_rows = []
table_results = []
pending_files = []
published_files = []

print()
print("-" * 70)
print("STARTING SOURCE EXTRACTION")
print("-" * 70)


try:

    with psycopg.connect(
        PG_URL,

        user=PG_USERNAME,
        password=PG_PASSWORD,

        connect_timeout=15,

        sslmode="require",

        row_factory=dict_row

    ) as connection:

        # Keep inventory and table reads consistent within one
        # read-only PostgreSQL transaction snapshot.
        with connection.cursor() as cursor:

            cursor.execute("""
                SET TRANSACTION
                ISOLATION LEVEL REPEATABLE READ,
                READ ONLY
            """)

            cursor.execute(
                TABLE_INVENTORY_QUERY
            )

            tables = [
                dict(row)
                for row in cursor.fetchall()
            ]

        if len(tables) != EXPECTED_TABLE_COUNT:
            raise RuntimeError(
                f"Expected {EXPECTED_TABLE_COUNT} tables; "
                f"discovered {len(tables)}."
            )

        missing_pk = [
            f"{row['table_schema']}.{row['table_name']}"
            for row in tables
            if not row["primary_key_columns"]
        ]

        if missing_pk:
            raise RuntimeError(
                f"Tables without primary keys: {missing_pk}"
            )

        print(
            f"Discovered source tables: {len(tables)}"
        )

        for index, table in enumerate(tables, start=1):

            schema_name = table["table_schema"]
            table_name = table["table_name"]

            primary_keys = table["primary_key_columns"]

            bronze_table = (
                f"pg_{schema_name}_{table_name}"
            )

            print(
                f"\n[{index}/{EXPECTED_TABLE_COUNT}] "
                f"{schema_name}.{table_name}"
            )

            # Build deterministic selection predicate.
            predicate = build_sample_predicate(
                primary_keys
            )

            query = (
                sql.SQL("SELECT * FROM {}.{} WHERE ").format(
                    sql.Identifier(schema_name),
                    sql.Identifier(table_name)
                )
                + predicate
            )

            table_directory = (
                f"{VOLUME_ROOT}/"
                f"{schema_name}/"
                f"{table_name}/"
                f"batch_id={BATCH_ID}"
            )

            os.makedirs(
                table_directory,
                exist_ok=True
            )

            final_output_path = (
                f"{table_directory}/part-00000.ndjson"
            )

            pending_output_path = (
                f"{final_output_path}.pending"
            )

            if (
                os.path.exists(final_output_path)
                or os.path.exists(pending_output_path)
            ):
                raise FileExistsError(
                    "Output path already exists for this batch: "
                    f"{final_output_path}"
                )

            extracted_count = 0

            try:

                with open(
                    pending_output_path,
                    "x",
                    encoding="utf-8"
                ) as output_file:

                    cursor_name = (
                        f"bp_extract_{index}_"
                        f"{uuid.uuid4().hex[:8]}"
                    )

                    with connection.cursor(
                        name=cursor_name
                    ) as cursor:

                        cursor.itersize = FETCH_SIZE

                        cursor.execute(
                            query,
                            (SAMPLE_PERCENT,)
                        )

                        while True:

                            rows = cursor.fetchmany(
                                FETCH_SIZE
                            )

                            if not rows:
                                break

                            for row in rows:

                                raw_record = dict(row)

                                source_pk = {
                                    key: raw_record[key]
                                    for key in primary_keys
                                }

                                envelope = {

                                    "_meta": {

                                        "batch_id": BATCH_ID,

                                        "environment": ENVIRONMENT,

                                        "source_system": "neon_postgresql",

                                        "source_database": DATABASE_NAME,

                                        "source_schema": schema_name,

                                        "source_table": table_name,

                                        "source_primary_key": json.dumps(
                                            source_pk,
                                            default=json_safe
                                        ),

                                        "snapshot_type": SNAPSHOT_TYPE,

                                        "sample_percent": SAMPLE_PERCENT,

                                        "ingested_at": (
                                            EXTRACTED_AT.isoformat()
                                        )
                                    },

                                    "record": raw_record
                                }

                                output_file.write(
                                    json.dumps(
                                        envelope,
                                        default=json_safe,
                                        ensure_ascii=False,
                                        separators=(",", ":")
                                    )
                                    + "\n"
                                )

                                extracted_count += 1

                    output_file.flush()
                    os.fsync(output_file.fileno())

            except Exception:

                # Remove the current incomplete file.
                try:
                    os.remove(pending_output_path)
                except FileNotFoundError:
                    pass

                raise

            pending_files.append({
                "pending_path": pending_output_path,
                "final_path": final_output_path
            })

            manifest_rows.append({

                "batch_id": BATCH_ID,

                "environment": ENVIRONMENT,

                "source_system": "neon_postgresql",

                "source_database": DATABASE_NAME,

                "source_schema": schema_name,

                "source_table": table_name,

                "bronze_table": bronze_table,

                "snapshot_type": SNAPSHOT_TYPE,

                "sample_percent": SAMPLE_PERCENT,

                "extracted_rows": extracted_count,

                "file_path": final_output_path,

                "extracted_at": EXTRACTED_AT
            })

            table_results.append({

                "source_schema": schema_name,

                "source_table": table_name,

                "bronze_table": bronze_table,

                "extracted_rows": extracted_count,

                "file_path": final_output_path
            })

            print(
                f"Extracted: {extracted_count:,} rows"
            )

        if len(manifest_rows) != EXPECTED_TABLE_COUNT:
            raise RuntimeError(
                "Extraction did not produce one result per expected table. "
                f"Expected {EXPECTED_TABLE_COUNT}; "
                f"got {len(manifest_rows)}."
            )

except Exception:

    # If extraction fails, remove every pending file from this batch.
    # No pending file matches the Bronze '*.ndjson' file filter.
    for item in pending_files:
        try:
            os.remove(item["pending_path"])
        except FileNotFoundError:
            pass

    raise


# ============================================================
# 8. PUBLISH COMPLETED FILES AND PERSIST MANIFEST
# ============================================================

MANIFEST_SCHEMA = StructType([

    StructField("batch_id", StringType(), False),

    StructField("environment", StringType(), False),

    StructField("source_system", StringType(), False),

    StructField("source_database", StringType(), False),

    StructField("source_schema", StringType(), False),

    StructField("source_table", StringType(), False),

    StructField("bronze_table", StringType(), False),

    StructField("snapshot_type", StringType(), False),

    StructField("sample_percent", IntegerType(), False),

    StructField("extracted_rows", LongType(), False),

    StructField("file_path", StringType(), False),

    StructField("extracted_at", TimestampType(), False)

])


manifest_data = [
    (
        row["batch_id"],
        row["environment"],
        row["source_system"],
        row["source_database"],
        row["source_schema"],
        row["source_table"],
        row["bronze_table"],
        row["snapshot_type"],
        row["sample_percent"],
        row["extracted_rows"],
        row["file_path"],
        row["extracted_at"]
    )
    for row in manifest_rows
]

manifest_df = spark.createDataFrame(
    manifest_data,
    schema=MANIFEST_SCHEMA
)


try:

    # Publish all completed NDJSON files only after every table
    # has been successfully extracted.
    for item in pending_files:

        os.rename(
            item["pending_path"],
            item["final_path"]
        )

        published_files.append(
            item["final_path"]
        )

    # Append one manifest row per extracted source table.
    (
        manifest_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(MANIFEST_TABLE)
    )

except Exception:

    # If publishing or manifest persistence fails, remove this
    # batch's files so an incomplete batch is not left for Bronze.
    for path in published_files:
        try:
            os.remove(path)
        except FileNotFoundError:
            pass

    for item in pending_files:
        try:
            os.remove(item["pending_path"])
        except FileNotFoundError:
            pass

    raise


# ============================================================
# 9. FINAL REPORT
# ============================================================

total_extracted = sum(
    row["extracted_rows"]
    for row in table_results
)

print()
print("=" * 70)
print("POSTGRESQL INITIAL SNAPSHOT COMPLETED")
print("=" * 70)

print("Environment:", ENVIRONMENT.upper())
print("Batch ID:", BATCH_ID)
print("Tables extracted:", len(table_results))
print("Total extracted rows:", f"{total_extracted:,}")
print("Sample percentage:", f"{SAMPLE_PERCENT}%")
print("Snapshot type:", SNAPSHOT_TYPE)
print("Manifest:", MANIFEST_TABLE)
print("Published NDJSON files:", len(published_files))
print()
print("SOURCE DATA WAS NOT MODIFIED.")
print(
    "IMPORTANT: Start or update the Bronze pipeline only "
    "after reviewing this extraction report."
)

display(
    spark.createDataFrame(table_results)
    .orderBy("source_schema", "source_table")
)